analises

In [36]:
from __future__ import annotations

import argparse
import csv
from collections import defaultdict
from pathlib import Path
from statistics import mean, median, stdev
import numpy as np
import pandas as pd
from datetime import datetime
import math

import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'src').exists():
    ROOT = (Path.cwd() / '..').resolve()
sys.path.insert(0, str(ROOT))

In [37]:

def carregar(caminho, columns, filter:dict = None):
    df = pd.read_csv(caminho, sep=',', header=0, usecols=columns)

    # sla se eu quiser aplicar algum filtro
    if filter:
        for key in filter:
            if key != "drop":
                df = df.loc[df[key] == filter[key]]
        if "drop" in filter: df = df.drop(columns=filter["drop"])

    #ordenação
    df = df.sort_values(by="ts_generation", ascending=True)
    # só pa ver :)
    df["date"] = pd.to_datetime(df["ts_generation"], unit="ms")
    # amostragem_ts(df)
    return df


def amostragem_ts(registro):
    # invervalos em segundos
    intervalos = [(registro["ts_generation"].iloc[i] - registro["ts_generation"].iloc[i-1])/1000 for i in range(1, registro.shape[0])]

    # entre a mediana
    intervalos_proximos_da_median = [
        intervalo for intervalo in intervalos if math.ceil(median(intervalos)-30) <= intervalo <= math.ceil(median(intervalos)+30)
    ]

    print(f"Intervalo mediano: {median(intervalos):.2f} s")
    print(f"Intervalo médio: ",f"{mean(intervalos):.2f} s ± {stdev(intervalos):.2f} s")
    print(f"Intervalos entre ± 30 s da mediana: {len(intervalos_proximos_da_median) / len(intervalos):.2%}", end="\n\n")

def tempo_proximo_evento(df):
    # ordenar por datatime...
    df = df.sort_values(by="ts_generation", ascending=True)
    
    # EVENTOS DE TRANSIÇÃO: 
    estado_anterior = df["valve_state"].shift(1)

    # 0 -> 1
    transicao = (
        estado_anterior.eq(0)
        & df["valve_state"].eq(1)
    )

    eventos = (df.loc[transicao].copy()).sort_values(by="ts_generation", ascending=True)

    eventos = eventos.drop(columns=["valve_state"])

    # TEMPO DE UM EVENTO DE IRRIGAÇÃO PARA OUTRO:
    ts = eventos["ts_generation"].to_numpy()

    # ts de i - (i-1) em s
    tempo_para_proxima = np.ceil((ts[1:] - ts[:-1]) / (60*1000))

    # tempo proxima irrigacao
    eventos["t_pi"] = [
        *tempo_para_proxima,
        float("nan")
    ]

    return eventos


In [38]:
ANO = "2024"
LINHA = 3
DISPOSITIVO_SOIL = "901b4b4b92f53023e62436410e7a000efb07750ba9bcb40327dd2a56a8259947"

ENTRADA_VALVE = ROOT / "data" / "raw" / ANO / f"valve_controller{ANO}.csv"
ENTRADA_ENVIRONMENTAL = ROOT / "data" / "raw" / ANO / f"environmental{ANO}.csv"
ENTRADA_INDICADOR = ROOT / "data" / "raw" / ANO / f"indicators{ANO}.csv"
ENTRADA_SOIL = ROOT / "data" / "raw" / ANO / f"soil{ANO}.csv"

COLUMNS_VALVE = {"ts_generation", "line", "valve_state"}
COLUMNS_ENV = {"ts_generation", "temperature", "humidity"}
COLUMNS_GDD = {"ts_generation", "gdd"}
COLUMNS_SOIL = {"ts_generation", "line", "humidity", "generated_by"}

FILTER_VALVE = {"line": LINHA, "drop": ["line"]}
FILTER_SOIL = {"line": LINHA, "generated_by": DISPOSITIVO_SOIL,  "drop": ["line", "generated_by"]} # filtrando por dispositivo 15cm


In [39]:

registros_valve = carregar(ENTRADA_VALVE, filter=FILTER_VALVE, columns=COLUMNS_VALVE)
print(registros_valve.shape)
registros_env = carregar(ENTRADA_ENVIRONMENTAL, columns=COLUMNS_ENV)
registros_env = registros_env.rename(columns={"temperature":"temperature_env","humidity": "humidity_env"})
print(registros_env.shape)
registros_gdd = carregar(ENTRADA_INDICADOR, columns=COLUMNS_GDD)
print(registros_gdd.shape)
registros_soil = carregar(ENTRADA_SOIL, filter=FILTER_SOIL, columns=COLUMNS_SOIL)
registros_soil = registros_soil.rename(columns={"humidity": "humidity_soil"})
print(registros_soil.shape)

(11658, 3)
(9327, 4)
(205, 3)
(13430, 3)


PROBLEMA: 
1. o sensoriamento tem intervalos médios e medianos próximos, mas estão desalinhados
2. União com merge retornar nenhum elemento, por causa das diferenças em ts

uniao com merge asof (vizinhos + exatos)

In [40]:
df_temp_1 = pd.merge_asof(registros_valve,registros_soil, on="ts_generation", direction="nearest", tolerance=10*60*1000)
print(tempo_proximo_evento(df_temp_1).shape)

# estado da valvula acaba sendo mais importante, então vou preservar ele
df_temp = pd.merge_asof(df_temp_1, registros_env, on="ts_generation", direction="nearest", tolerance=10*60*1000)
print(tempo_proximo_evento(df_temp).shape)

df_temp = df_temp.dropna()
df_temp = df_temp.drop(columns=["date", "date_x", "date_y"])
df_temp.shape

(74, 5)
(74, 8)


(9517, 5)

autoregressivo

Regras de composição do data (preguiça t_T): ### NAO APLICADO 
1. i, se v_i = 0 e v_j = 0 para todo j pertence (i-10,i-20,i-30)
2. i, se v_i = 1 e umidade_j != Null para todo j pertence (i-10,i-20,i-30)
composição deve ter:
* umid_solo; umid_ar; temp_ar; ts; valve_state; umid_solo(t-10); umid_solo(t-20); umid_solo(t-30); t_i (tempo de irrigação)

In [43]:
df_aut = df_temp.copy()
df_aut[['humidity_soil_1', 'humidity_soil_2']] =  df_temp["humidity_soil"].shift(periods=[1, 2])
df_aut = df_aut.dropna()
df_aut.head()

,ts_generation,valve_state,humidity_soil,temperature_env,humidity_env,humidity_soil_1,humidity_soil_2
2,1718152848858,0.0,0.0,17.1,91.0,0.0,0.0
3,1718154048848,0.0,0.0,16.7,90.0,0.0,0.0
4,1718155248822,0.0,0.0,16.5,92.0,0.0,0.0
5,1718156448819,0.0,0.0,16.3,96.0,0.0,0.0
6,1718157648806,0.0,0.0,16.2,100.0,0.0,0.0


In [46]:
import skfuzzy as fuzz
from skfuzzy import control as ctrl

ModuleNotFoundError: No module named 'networkx'

In [10]:
df_aut.shape

(9514, 8)

**obs1**: foi observado que uma data da df_2 pode ser pareada com mais do que uma df_1, dado que ele faz um how left


> **talvez futuro**: algo minimize a diferença de tempo entre os pares, em que 
> 1. Cada registro da df_1 pode ser associado a apenas um registro da df_2.
> 2. Cada registro da df_2 só pode ser utilizado uma vez.
> 3. com uma tolerancia em minutos. 

REMOÇÃO OUTLIERS

In [7]:
from src.utils.pca import pca, t2_q, remove_anomalias_t2_q
from sklearn.model_selection import train_test_split

seed = 42

def outlier_iqr(
    df: pd.DataFrame,
    columns: list | None = None,
    show: bool = True,
    remove: bool = False
):
    if columns is None:
        columns = df.columns

    df_iqr = df.copy()

    contagem_por_coluna = {}
    mask_remover = pd.Series(False, index=df_iqr.index)

    for column in columns:
        q1 = df_iqr[column].quantile(0.25)
        q3 = df_iqr[column].quantile(0.75)

        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        mask_outlier = (
            (df_iqr[column] < lower) |
            (df_iqr[column] > upper)
        )

        outlier_num = mask_outlier.sum()

        contagem_por_coluna[column] = outlier_num
        mask_remover |= mask_outlier

        if show:
            print(
                f"Quantidade de outliers da coluna "
                f"{column}: {outlier_num}"
            )

    idx_removidos = df_iqr.index[mask_remover]

    if remove:
        df_iqr = df_iqr.loc[~mask_remover].copy()

    return df_iqr, idx_removidos, contagem_por_coluna

def clean_anormaly(df, PARAMS, cols_meta, cols_target, flag_pca: bool = True, cutoff: str = None, save_flag: bool =  False, sufix_file = "none", name_dir = "none" ):
    cols_features = [c for c in df.columns if c not in cols_meta + cols_target]

    # treino e teste
    if cutoff is None:
        train_idx, test_idx = train_test_split(df.index, test_size=0.3, random_state=seed, shuffle=False)
    # sem split temporal por enquanto...

    train_df = df.loc[train_idx, cols_meta + cols_features + cols_target]
    df_test  = df.loc[test_idx,  cols_meta + cols_features + cols_target]

    # iqr
    train_df, idx_removidos_iqr, contagem_por_coluna = outlier_iqr(
        train_df.copy(),
        PARAMS,
        remove=True,
        show=False
    )

    ## REMOVENDO AUSENTES!!!
    train_df = train_df.dropna()

    print("\n================ IQR ================")
    print(f"Total de linhas removidas pelo IQR: {len(idx_removidos_iqr)}")

    print("\nColunas responsáveis pela remoção:")

    for col, qtd in contagem_por_coluna.items():
        if qtd > 0:
            print(f"{col}: {qtd} valores fora do IQR")

    if flag_pca:
        X_train = train_df[PARAMS]
        meta_train = train_df[cols_meta]
        cols_restantes = train_df.columns.difference(PARAMS + cols_meta).tolist()
        y_train = train_df[cols_restantes]

        # aplicando pca
        vals, vecs, scaler, Z_train = pca(X_train=X_train)

        # aplicando t2q
        res = t2_q(Z_train, vals, vecs, k=3, name_IMG=sufix_file)

        # removendo anomalias t2q
        X_train_clean, y_train_clean, meta_train_clean = remove_anomalias_t2_q(X_train, y_train, res, meta_train)
        train_df = pd.concat([meta_train_clean, X_train_clean, y_train_clean], axis=1)

    # if save_flag:
    #     save_file(train_df, name=f"train_{sufix_file}", name_dir=name_dir)
    #     save_file(df_test, name=f"test_{sufix_file}", name_dir=name_dir)

    return train_df, df_test


In [8]:
PARAMS = [
    'humidity_soil',
    'temperature_env',
    'humidity_env',
]

cols_meta = ['ts_generation']
cols_target = ['t_pi']

clean_anormaly(df_aut, PARAMS=PARAMS, cols_meta=cols_meta, cols_target=cols_target)

KeyError: "['t_pi'] not in index"

In [ ]:
import matplotlib.pyplot as plt
df_aut_t = df_aut.copy()

# # quando humidity_soil != 0:
df_aut_t["date"] = pd.to_datetime(df_aut_t["ts_generation"], unit="ms")
mask = df_aut_t["humidity_soil"].ne(0.0)
date = df_aut_t.loc[mask, "date"].iloc[0]

print(date)
df_aut_t = df_aut_t[df_aut_t["date"] > date]

# # plt.plot(df_aut_t["humidity_soil"])
plt.boxplot(df_aut_t["humidity_soil"])
plt.show()

# tendencia
df_aut_t["tend"] = df_aut_t["humidity_soil"] - df_aut_t["humidity_soil_3"]
# df_aut_t.head(5)

plt.hist(df_aut_t["tend"])
# plt.xlim(-5,5)
plt.show()

MODEL